In [11]:
import neo4j from 'neo4j-driver'

const driver = neo4j.driver(
  'bolt://localhost:7687',
  neo4j.auth.basic('neo4j', 'password'),
  { disableLosslessIntegers: true },
)
const session = driver.session()

await session.run('MATCH (n) DETACH DELETE n')
await session.run(`
  CREATE
    (:Video {name: "long", interviewID: 26}),
    (:Video {name: "kramer", interviewID: 35}),
    (:Video {name: "crimp", interviewID: 74})
`)

{
  records: [],
  summary: ResultSummary {
    query: {
      text: '\n' +
        '  CREATE\n' +
        '    (:Video {name: "long", interviewID: 26}),\n' +
        '    (:Video {name: "kramer", interviewID: 35}),\n' +
        '    (:Video {name: "crimp", interviewID: 74})\n',
      parameters: {}
    },
    queryType: 'w',
    counters: QueryStatistics {
      _stats: [Object],
      _systemUpdates: 0,
      _containsUpdates: true
    },
    updateStatistics: QueryStatistics {
      _stats: [Object],
      _systemUpdates: 0,
      _containsUpdates: true
    },
    plan: false,
    profile: false,
    notifications: [],
    server: ServerInfo {
      address: 'localhost:7687',
      agent: 'Neo4j/5.8.0',
      protocolVersion: 5.2
    },
    resultConsumedAfter: 0,
    resultAvailableAfter: 1,
    database: { name: 'neo4j' }
  }
}


In [12]:
import { Session } from 'neo4j-driver'

const createIndexQuery = `
  CREATE FULLTEXT INDEX search_interview_lines IF NOT EXISTS
  FOR (l:Line) ON EACH [l.text]
  OPTIONS {
    indexConfig: {
      \`fulltext.analyzer\`: 'english'
    }
  }
`

try {
  await session.run(createIndexQuery)
} catch (err) {
  console.error(err)
}

In [13]:
import fs from 'fs/promises'
import path from 'path'

const TRANSCRIPTION_JSON = [
  '026-000.json',
  '026-001.json',
  '026-002.json',
  '026-003.json',
  '035-000.json',
  '035-001.json',
  '035-002.json',
  '074-000.json',
  '074-001.json',
  '074-002.json',
  '074-003.json',
  '074-004.json',
]

for await (const jsonFile of TRANSCRIPTION_JSON) {
  const transcriptData = await fs.readFile(path.join('../assets', jsonFile), 'utf-8')
  const transcript = JSON.parse(transcriptData)

  // const transcribedItems = transcript.results.items.filter((item) => item.speaker_label === 'spk_2')
  const transcribedItems = transcript.results.items
  const lines = []

  const CAPTION_WORDS = 12

  for (let i = 0; i < transcribedItems.length; i += CAPTION_WORDS) {
    const items = transcribedItems.slice(i, i + CAPTION_WORDS)
    const text = items.map((item) => item.alternatives[0].content).join(' ')
    lines.push({
      text,
      startTime: Number.parseFloat(items[0].start_time),
      endTime: Number.parseFloat(items[items.length - 1].end_time),
    })
  }
  
  const iid = Number.parseInt(jsonFile.slice(0, 3), 10)

  // TK: Figure out `MERGE`.
  const createLinesQuery = `
    MATCH (v:Video {interviewID: $iid})
    WITH v
    UNWIND $lines AS line
    CREATE (l:Line {text: line.text, startTime: duration({ seconds: line.startTime }), endTime: duration({ seconds: line.endTime })}),
    (v)-[:HAS_LINE {startTime: duration({ seconds: line.startTime }), endTime: duration({ seconds: line.endTime }) }]->(l)
  `

  const { summary } = await session.run(createLinesQuery, { iid, lines })
  console.log(`
    Created ${ summary.counters.updates().nodesCreated } nodes
    and ${ summary.counters.updates().relationshipsCreated } relationships
    for ${ jsonFile }
    in ${ summary.resultAvailableAfter } ms
  `)
}

35.229

    Created 368 nodes
    and 368 relationships
    for 026-000.json
    in 24 ms
  
0.009

    Created 397 nodes
    and 397 relationships
    for 026-001.json
    in 3 ms
  
0.009

    Created 380 nodes
    and 380 relationships
    for 026-002.json
    in 3 ms
  
0.009

    Created 213 nodes
    and 213 relationships
    for 026-003.json
    in 2 ms
  
4.23

    Created 516 nodes
    and 516 relationships
    for 035-000.json
    in 3 ms
  
0.009

    Created 567 nodes
    and 567 relationships
    for 035-001.json
    in 4 ms
  
0.009

    Created 436 nodes
    and 436 relationships
    for 035-002.json
    in 3 ms
  
30.02

    Created 359 nodes
    and 359 relationships
    for 074-000.json
    in 2 ms
  
0.009

    Created 335 nodes
    and 335 relationships
    for 074-001.json
    in 3 ms
  
0.009

    Created 371 nodes
    and 371 relationships
    for 074-002.json
    in 3 ms
  
0.009

    Created 338 nodes
    and 338 relationships
    for 074-003.json
    in 3 ms
 

In [14]:
const executeSearchQuery = `
  CALL db.index.fulltext.queryNodes('search_interview_lines', 'yale studies') YIELD node AS lineNode, score
  WITH lineNode, score
  MATCH (v:Video) -[:HAS_LINE]-> (lineNode) 
  RETURN lineNode.text, lineNode.startTime, lineNode.endTime, v.name, score
`

const { records } = await session.run(executeSearchQuery)
console.log(records.map((record) => record.toObject()))

[
  {
    'lineNode.text': 'Larry Kramer initiative for lesbian and gay Studies at Yale Endowed by',
    'lineNode.startTime': Duration {
      months: 0,
      days: 0,
      seconds: [Integer],
      nanoseconds: [Integer]
    },
    'lineNode.endTime': Duration {
      months: 0,
      days: 0,
      seconds: [Integer],
      nanoseconds: [Integer]
    },
    'v.name': 'kramer',
    score: 4.881775379180908
  },
  {
    'lineNode.text': 'to get Yale to teach gay studies , offering them money along',
    'lineNode.startTime': Duration {
      months: 0,
      days: 0,
      seconds: [Integer],
      nanoseconds: [Integer]
    },
    'lineNode.endTime': Duration {
      months: 0,
      days: 0,
      seconds: [Integer],
      nanoseconds: [Integer]
    },
    'v.name': 'kramer',
    score: 4.6217122077941895
  },
  {
    'lineNode.text': ', who was this for gay and lesbian studies studies ? No',
    'lineNode.startTime': Duration {
      months: 0,
      days: 0,
      seconds: [Inte